# Chris Camillo — Social Arbitrage Research Notebook

**Based on:** *Laughing at Wall Street* by Chris Camillo

**Thesis:** Markets are efficient at processing *financial* information but slow to price *cultural* information. The gap between everyday consumer observation and institutional analyst coverage is the edge.

---

## Workflow
1. Define your trend thesis
2. Validate with Google Trends
3. Check analyst coverage gap
4. Measure price lag
5. Score and decide

In [ ]:
# Install dependencies
# !pip install yfinance pytrends pandas numpy matplotlib vaderSentiment praw

In [ ]:
import warnings
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
from pytrends.request import TrendReq
from datetime import datetime

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

pytrends = TrendReq(hl='en-US', tz=360, timeout=(10, 30))
print('Libraries loaded.')

---
## Step 1 — Define Your Thesis

Fill in the cell below. The keywords should be what a non-investor would type into Google — not the company name or ticker.

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────
TICKER   = 'CELH'                              # Stock ticker
KEYWORDS = ['celsius drink', 'celsius energy', 'celsius gym']  # Cultural search terms
THESIS   = 'Celsius energy drink is replacing Red Bull/Monster in gyms and convenience stores.'
# ────────────────────────────────────────────────────────────────

print(f'Ticker:  {TICKER}')
print(f'Thesis:  {THESIS}')
print(f'Keywords: {KEYWORDS}')

---
## Step 2 — Google Trends: Is the Cultural Trend Accelerating?

In [ ]:
def fetch_trends(keywords, timeframe='today 12-m'):
    pytrends.build_payload(keywords[:5], timeframe=timeframe)
    df = pytrends.interest_over_time()
    time.sleep(0.6)
    if 'isPartial' in df.columns:
        df = df.drop(columns=['isPartial'])
    return df

trends_df = fetch_trends(KEYWORDS)
print(f'Trend data: {len(trends_df)} weeks')
trends_df.tail()

In [ ]:
def trend_velocity(df, primary_kw):
    col    = primary_kw if primary_kw in df.columns else df.columns[0]
    series = df[col].values.astype(float)
    x      = np.arange(len(series))
    slope, intercept = np.polyfit(x, series, 1)
    trend_line       = slope * x + intercept
    
    recent = series[-4:].mean()
    prior  = series[-8:-4].mean() if len(series) >= 8 else series[:4].mean()
    accel  = (recent - prior) / (prior + 1e-9)
    
    return slope, accel, trend_line, col

slope, accel, trend_line, primary_col = trend_velocity(trends_df, KEYWORDS[0])

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(trends_df.index, trends_df[primary_col], alpha=0.3, color='steelblue')
ax.plot(trends_df.index, trends_df[primary_col], color='steelblue', lw=1.5, label='Search Interest')
ax.plot(trends_df.index, trend_line, color='tomato', lw=2, linestyle='--', label=f'Trend (slope={slope:.2f}/wk)')
ax.set_title(f'Google Trends — "{KEYWORDS[0]}"', fontsize=13)
ax.set_ylabel('Interest (0–100)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend()
plt.tight_layout()
plt.show()

print(f'Slope:        {slope:+.3f} per week  ({"rising" if slope > 0 else "falling"})')
print(f'Acceleration: {accel:+.1%}  (last 4wk vs prior 4wk)')
slope_score = float(np.clip(slope / 2.0 * 100, 0, 100))
accel_score = float(np.clip(accel * 100, 0, 100))
trend_score = round(slope_score * 0.5 + accel_score * 0.5, 1)
print(f'Trend Score:  {trend_score}/100')

---
## Step 3 — Analyst Coverage Gap

In [ ]:
stock = yf.Ticker(TICKER)
info  = stock.info

count   = info.get('numberOfAnalystOpinions', 0) or 0
target  = info.get('targetMeanPrice')
price   = info.get('currentPrice') or info.get('regularMarketPrice', 0)
upside  = (target - price) / price * 100 if (target and price) else 0
rec     = info.get('recommendationKey', 'n/a')

print(f'Company:          {info.get("longName", TICKER)}')
print(f'Current Price:    ${price:.2f}')
print(f'Analyst Count:    {count}')
print(f'Consensus Target: ${target:.2f}  ({upside:+.1f}% upside)' if target else 'No target')
print(f'Recommendation:   {rec}')

if   count == 0:  cov_score = 100
elif count <= 2:  cov_score = 90
elif count <= 5:  cov_score = 75
elif count <= 10: cov_score = 55
elif count <= 20: cov_score = 30
else:             cov_score = 10

upside_bonus  = float(np.clip(upside / 50 * 20, 0, 20))
analyst_score = round(min(cov_score + upside_bonus, 100), 1)
print(f'\nAnalyst Gap Score: {analyst_score}/100')
print('  (Higher = fewer analysts = larger information gap = more Camillo edge)')

In [ ]:
# Analyst recommendation breakdown
try:
    recs = stock.recommendations_summary
    if recs is not None and not recs.empty:
        latest = recs.iloc[0]
        cats   = ['strongBuy', 'buy', 'hold', 'sell', 'strongSell']
        vals   = [latest.get(c, 0) for c in cats]
        colors = ['#2ecc71', '#82e0aa', '#f39c12', '#e74c3c', '#922b21']
        fig, ax = plt.subplots(figsize=(8, 3))
        bars = ax.barh(cats, vals, color=colors)
        ax.set_xlabel('Number of Analysts')
        ax.set_title(f'{TICKER} — Analyst Recommendation Breakdown')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                    str(int(v)), va='center')
        plt.tight_layout()
        plt.show()
except Exception:
    print('Recommendation breakdown not available.')

---
## Step 4 — Price Lag: Has the Stock Moved Yet?

In [ ]:
hist = stock.history(period='1y')

prices  = hist['Close'].values
px_norm = prices / prices[0]
px_x    = np.arange(len(px_norm))
price_slope, _ = np.polyfit(px_x, px_norm, 1)

# Normalize trend to same length for comparison
trend_52w = fetch_trends(KEYWORDS, timeframe='today 12-m')
col       = KEYWORDS[0] if KEYWORDS[0] in trend_52w.columns else trend_52w.columns[0]
t_vals    = trend_52w[col].values.astype(float)
t_norm    = t_vals / (t_vals.mean() + 1e-9)
t_x       = np.arange(len(t_norm))
trend_slope52, _ = np.polyfit(t_x, t_norm, 1)

gap = trend_slope52 - price_slope
lag_score = float(np.clip(gap * 600, 0, 100))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=False)

ax1.plot(hist.index, px_norm * 100, color='steelblue', lw=1.5)
ax1.axhline(100, color='gray', lw=0.8, linestyle=':')
ax1.set_title(f'{TICKER} — Normalized Price (1Y)', fontsize=11)
ax1.set_ylabel('Price (base=100)')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

ax2.fill_between(trend_52w.index, trend_52w[col], alpha=0.3, color='darkorange')
ax2.plot(trend_52w.index, trend_52w[col], color='darkorange', lw=1.5)
ax2.set_title(f'Google Trends — "{KEYWORDS[0]}" (1Y)', fontsize=11)
ax2.set_ylabel('Search Interest (0–100)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.tight_layout()
plt.show()

px_3mo = (prices[-1] / prices[-63] - 1) * 100 if len(prices) >= 63 else (prices[-1]/prices[0]-1)*100
print(f'Price 3-month change:  {px_3mo:+.1f}%')
print(f'Trend slope (52w):     {trend_slope52:+.5f}')
print(f'Price slope (52w):     {price_slope:+.5f}')
print(f'Arbitrage Gap:         {gap:+.5f}')
print(f'Price Lag Score:       {lag_score:.1f}/100')
print()
if gap > 0:
    print('→ Trend outpacing price. Potential Camillo setup.')
else:
    print('→ Price has moved with or ahead of trend. Edge reduced.')

---
## Step 5 — Composite Score & Decision

In [ ]:
WEIGHTS = {'trend': 0.30, 'analyst': 0.25, 'reddit': 0.25, 'price_lag': 0.20}

# Reddit score: enter manually or run the full scanner
reddit_score = 50.0  # default neutral if not configured
print('Reddit score defaulted to 50 (neutral). Configure PRAW to enable.')
print('See camillo_social_arbitrage.py for Reddit setup instructions.\n')

scores = {
    'Trend Velocity':  trend_score,
    'Analyst Gap':     analyst_score,
    'Reddit Buzz':     reddit_score,
    'Price Lag':       lag_score,
}

composite = round(
    trend_score   * WEIGHTS['trend']
    + analyst_score * WEIGHTS['analyst']
    + reddit_score  * WEIGHTS['reddit']
    + lag_score     * WEIGHTS['price_lag'],
    1
)

signal = 'BUY' if composite >= 70 else 'WATCH' if composite >= 50 else 'PASS'

# Score breakdown chart
fig, ax = plt.subplots(figsize=(8, 4))
factor_names  = list(scores.keys())
factor_vals   = list(scores.values())
factor_colors = ['#2ecc71' if v >= 70 else '#f39c12' if v >= 50 else '#e74c3c' for v in factor_vals]
bars = ax.barh(factor_names, factor_vals, color=factor_colors, height=0.5)
ax.axvline(70, color='green',  lw=1.5, linestyle='--', alpha=0.7, label='Buy threshold (70)')
ax.axvline(50, color='orange', lw=1.5, linestyle='--', alpha=0.7, label='Watch threshold (50)')
ax.set_xlim(0, 100)
ax.set_xlabel('Score (0–100)')
ax.set_title(f'{TICKER} — Social Arbitrage Factor Scores', fontsize=12)
ax.legend(loc='lower right')
for bar, v in zip(bars, factor_vals):
    ax.text(v + 1, bar.get_y() + bar.get_height()/2, f'{v:.1f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

print('═' * 50)
print(f'  TICKER:     {TICKER}')
print(f'  COMPOSITE:  {composite}/100')
print(f'  SIGNAL:     {signal}')
print('─' * 50)
for k, v in scores.items():
    print(f'  {k:<18} {v:>5.1f}/100')
print('═' * 50)

---
## Step 6 — Pre-Trade Checklist

In [ ]:
checklist = [
    ('I can describe the trend in plain English (not financial jargon)',      None),
    ('I observed this trend personally OR heard it from a non-investor',      None),
    ('I can verify the trend in the real world (try the product/app/place)',  None),
    ('Analyst count is low (institutional blind spot exists)',                count <= 10),
    ('Composite score is >= 70',                                              composite >= 70),
    ('Price has not already run > 30% in the past 3 months',                 abs(px_3mo) < 30),
    ('I have a clear exit scenario defined',                                  None),
]

print(f'PRE-TRADE CHECKLIST — {TICKER}\n')
for item, auto_check in checklist:
    if auto_check is True:
        status = '✓ AUTO'
    elif auto_check is False:
        status = '✗ FAIL'
    else:
        status = '□ MANUAL'
    print(f'  [{status}]  {item}')

print()
print('Items marked [□ MANUAL] require your personal judgment.')
print('Camillo: do not enter if you cannot check every box honestly.')

---
## Step 7 — Run the Full Scanner (Multi-Ticker)

In [ ]:
# Runs the camillo_social_arbitrage.py scanner on the full watchlist
# Adjust the watchlist below before running

import sys
import importlib.util

WATCHLIST = [
    {'ticker': 'CELH', 'keywords': ['celsius drink', 'celsius energy drink']},
    {'ticker': 'ONON', 'keywords': ['on running shoes', 'on cloud shoes']},
    {'ticker': 'DUOL', 'keywords': ['duolingo', 'duolingo streak']},
    {'ticker': 'BIRK', 'keywords': ['birkenstock', 'birkenstock sandals']},
]

spec   = importlib.util.spec_from_file_location('camillo', 'camillo_social_arbitrage.py')
mod    = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

scanner = mod.SocialArbitrageScanner(reddit_creds=None)
results = scanner.scan_watchlist(WATCHLIST)
results

In [ ]:
# Visual comparison across tickers
if not results.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    x     = np.arange(len(results))
    width = 0.18
    cols  = ['Trend', 'AnalystGap', 'Reddit', 'PriceLag']
    clrs  = ['steelblue', 'darkorange', 'mediumseagreen', 'tomato']
    for i, (col, clr) in enumerate(zip(cols, clrs)):
        ax.bar(x + i * width, results[col], width, label=col, color=clr, alpha=0.85)
    ax.axhline(70, color='green',  lw=1.2, linestyle='--', alpha=0.6, label='Buy (70)')
    ax.axhline(50, color='orange', lw=1.2, linestyle='--', alpha=0.6, label='Watch (50)')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(results['Ticker'])
    ax.set_ylabel('Score (0–100)')
    ax.set_title('Social Arbitrage Factor Scores — Watchlist Comparison')
    ax.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.show()